# Bibliometric pipeline — run everything, one script per cell

Open this notebook from the package folder (the one that contains `run_all.py`). Put the raw Web of Science exports in `data/TS-J`, `data/AS-J`, `data/AS-T` first. Each cell runs one script and shows its full printed output. The last cell compares every analysis output with the deposited copy.

If you only want to redraw the figures from the deposited outputs, skip the four *analysis* cells.

In [ ]:
import os, subprocess, sys, shutil, filecmp
HERE = os.getcwd()
SCRIPTS = os.path.join(HERE, 'scripts')
assert os.path.isdir(SCRIPTS), 'run this notebook from the package folder'
if not os.path.isdir('analysis_outputs_frozen'):
    shutil.copytree('analysis_outputs', 'analysis_outputs_frozen')
    print('deposited outputs preserved in analysis_outputs_frozen/')

def run(script):
    print(f'$ python3 scripts/{script}\n')
    p = subprocess.run([sys.executable, script], cwd=SCRIPTS, capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr)
        raise SystemExit(f'{script} failed')


## Analysis 1 — parse the 73 exports; audit; term presence; canon matching; ledger; baselines; citation kind; noise sample; frontier/core; clocks; RPYS; founders

In [ ]:
run('analysis_cross_corpus.py')

## Analysis 2 — token-level markedness, rate-normalised clocks, windowed RPYS

In [ ]:
run('analysis_refined.py')

## Analysis 3 — keyword networks and thematic maps of TS-J and AS-J (Louvain, seed 20260806)

In [ ]:
run('analysis_networks.py')

## Analysis 4 — the same procedure over all three corpora in one pass (Figures 6.4–6.6)

In [ ]:
run('analysis_three_corpora.py')

## Keep the deposited Louvain partitions for the figures

Community detection differs between `networkx` versions (README, D19). If the fresh partition differs from the deposited one, it is kept beside it and the deposited partition is restored so the maps match the thesis.

In [ ]:
import networkx
for f in ('NETWORK_RESULTS.json', 'THREE_CORPORA_THEMES.json'):
    a, b = os.path.join('analysis_outputs', f), os.path.join('analysis_outputs_frozen', f)
    if os.path.exists(b) and not filecmp.cmp(a, b, shallow=False):
        keep = a[:-5] + f'_rerun_networkx{networkx.__version__}.json'
        shutil.move(a, keep); shutil.copy(b, a)
        print(f, '-> partition differs under networkx', networkx.__version__, '; kept as', os.path.basename(keep))
    else:
        print(f, 'identical to the deposited partition')

## Figures 6.1, 6.7–6.13, 6.15

In [ ]:
run('make_figures_palette.py')

## Figure A.1

In [ ]:
run('make_fig_a1_rpys.py')

## Figures 6.4, 6.5, 6.6

In [ ]:
run('make_labelled_maps.py')

## Three-panel map (not used in the thesis)

In [ ]:
run('make_three_maps.py')

## Figures 1.1, 1.2, 5.1, 6.2, 6.3, 6.14, 6.16, 6.17

In [ ]:
run('make_chapter_figures.py')

## Figure A.2

In [ ]:
run('make_fig_a2_flow.py')

## Tables 6.1, 6.2, 6.4, 6.5, A.2 (+ S.1, S.2) as CSV

In [ ]:
run('make_tables.py')

## First figure set (superseded)

In [ ]:
run('make_figures.py')

## Descriptive figures (superseded)

In [ ]:
run('make_descriptive_figures.py')

## Compare every analysis output with the deposited copy

In [ ]:
names = sorted(f for f in os.listdir('analysis_outputs_frozen') if f.endswith(('.json', '.csv')) and 'rerun' not in f)
for f in names:
    fresh = [g for g in os.listdir('analysis_outputs') if g.startswith(f[:-5] + '_rerun_')] if f.endswith('.json') else []
    a = os.path.join('analysis_outputs', fresh[0] if fresh else f)
    same = filecmp.cmp(a, os.path.join('analysis_outputs_frozen', f), shallow=False)
    print('IDENTICAL ' if same else 'DIFFERS   ', os.path.basename(a))

## Look at the figures

Every thesis figure, by its thesis number, is in `figures_as_in_thesis/` after `run_all.py`; the raw script outputs are in `figs_out/`.

In [ ]:
from IPython.display import Image, display
for f in ['figs_out/palette/fig_6_03_indexed_output.png', 'figs_out/thematic_maps/fig_themes_TSJ.png', 'figs_out/fig_5_03_thematic_evolution.png', 'figs_out/fig_5_04_geography.png']:
    print(f); display(Image(f, width=700))